# Real-Time Log Simulation for AI-SOC

In [ ]:
import pandas as pd
import numpy as np
import joblib
import time
from datetime import datetime

In [ ]:
rf = joblib.load("../models/random_forest.pkl")

In [ ]:
df = pd.read_csv("../data/processed/unsw_nb15_processed.csv")

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["label"])
y = df["label"]

_, X_stream, _, y_stream, = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
def soc_detect(event, model, threshold=0.7):
    prob = model.predict_proba(event.values.reshape(1, -1))[0][1]

    if prob >= threshold:
        return "ALERT", prob
    else:
        return "NORMAL", prob

In [ ]:
alerts = []

for i in range(100):
    event = X_stream.iloc[i]
    actual = y_stream.iloc[i]

    status, score = soc_detect(event, rf)

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"[{timestamp}] Event {i} | Score={score:.2f} | Status={status}")

    if status == "ALERT":
        alerts.append({
            "event_id": i,
            "risk_score": score,
            "actual_label": actual,
            "time": timestamp
        })

    time.sleep(0.5)

In [ ]:
alerts_df = pd.DataFrame(alerts)
alerts_df